# Data Understanding

This notebook explores the two datasets `freMTPL2freq` and `freMTPL2sev` fetched from OpenML. They contain data on motor third-party liability policies such as the claim number, the claim amount, and certain other risk features. To understand the data, we check the size of both tables and how they link via the policy ID `IDpol`. We also look at value ranges, missing values, and inconsistencies that need cleaning before modeling.

### Data Loading

We first load the data using the functions `load_freq` and `load_sev` from `src.data`, which fetch the two datasets from OpenML and cache the raw data in `data/`.

In [51]:
from src.data import load_freq, load_sev
import numpy as np
import pandas as pd
from pyprojroot import here

%load_ext autoreload
%autoreload 2

DATA_PATH = here("data")
ID_COL = "IDpol"

# Load the datasets using DATA_PATH as the cache location
freq_df = load_freq(DATA_PATH)
sev_df = load_sev(DATA_PATH)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Dataset Size

We first check how many rows and columns each table has.

In [52]:
# Get the row and column numbers
num_rows_freq = freq_df.shape[0]
num_rows_sev = sev_df.shape[0]
num_cols_freq = freq_df.shape[1]
num_cols_sev = sev_df.shape[1]

# Print results
print(f"Shape of freq_df: {num_rows_freq} rows, {num_cols_freq} columns")
print(f"Shape of sev_df: {num_rows_sev} rows, {num_cols_sev} columns")

Shape of freq_df: 678013 rows, 12 columns
Shape of sev_df: 26639 rows, 2 columns


### 2. Key Columns

Since `IDpol` is the only column both tables share, `IDpol` is the key by which both tables are linked by it. Thus, we now check the data type of `IDpol` in both tables and whether it identifies each row uniquely. 

In [53]:
# Store the policy IDs in their own pd.Series
freq_ids = freq_df[ID_COL].astype("int64")
sev_ids = sev_df[ID_COL].astype("int64")

# Count unique values in both tables
num_pols_unique_freq = freq_ids.nunique()
num_pols_unique_sev = sev_ids.nunique()

# Count rows that repeat ´IDpol´ in sev_df
sev_id_counts = sev_ids.value_counts()
sev_repeated_ids =sev_id_counts[sev_id_counts > 1]

# Print results
print(f"dtype of IDpol in freq_df: {freq_df[ID_COL].dtypes}")
print(f"dtype of IDpol in sev_df: {sev_df[ID_COL].dtypes}")
print()
print(f"Unique {ID_COL} in freq_df: {num_pols_unique_freq} (out of {len(freq_ids)}), all unique: {freq_ids.is_unique}")
print(f"Unique {ID_COL} in sev_df: {num_pols_unique_sev} (out of {len(sev_ids)}), all unique: {sev_ids.is_unique}")
print()
print(f"Number of rows in sev_df that contain repeated {ID_COL}: {len(sev_ids) - sev_ids.nunique()}")
print(f"{ID_COL} values that appear more than once in sev_df: {len(sev_repeated_ids)}")
print(f"{ID_COL} that appears the most frequent in sev_df: {sev_id_counts.idxmax()} ({sev_id_counts.max()} times)")

dtype of IDpol in freq_df: float64
dtype of IDpol in sev_df: int64

Unique IDpol in freq_df: 678013 (out of 678013), all unique: True
Unique IDpol in sev_df: 24950 (out of 26639), all unique: False

Number of rows in sev_df that contain repeated IDpol: 1689
IDpol values that appear more than once in sev_df: 1379
IDpol that appears the most frequent in sev_df: 2262511 (66 times)


**Insights** 
- `IDpol` is unique in `freq_df` (678013 unique values in 678013 rows) and not unique in `sev_df` (24950 unique values in 26639 rows). Thus, `IDpol` is only a true key in `freq_df`.
- There are 1689 rows in `sev_df` that repeat IDs and 1379 IDs that appear more than once. Note that both numbers are different since some IDs appear more than twice (e.g., ID 2262511 appears 66 times). Thus, rows and unique IDs are different units and we keep them apart in the following.

### 3. Claim Counts Across Tables

We continue by counting how many policies and claims the frequency table contains, comparing these totals with the rows and policies in the severity table, and looking at where the two tables do not match.

In [54]:
# Define two masks for filtering claims
claimed = freq_df["ClaimNb"] > 0
claimed_mult = freq_df["ClaimNb"] > 1

# Get the IDs and the number of policies with no claims
pols_not_claimed = freq_ids[~claimed]
num_pols_not_claimed = int(len(pols_not_claimed))

# Get the IDs and the number of policies with at least one claim
pols_claimed = freq_ids[claimed]
num_pols_claimed = int(claimed.sum())

# Get the IDs and the number of policies with multiple claims
pols_claimed_mult = freq_ids[claimed_mult]
num_pols_claimed_mult = int(len(pols_claimed_mult))

# Compute the total number of claims
total_claims = int(freq_df["ClaimNb"].sum())

# Print results
print(f"Policies with no claim (ClaimNb = 0): {num_pols_not_claimed}")
print(f"Policies with at least one claim (ClaimNb > 0): {num_pols_claimed}")
print(f"Policies with multiple claims (ClaimNb > 1): {num_pols_claimed_mult}")
print(f"Total claims (sum of ClaimNb): {total_claims}")
print()
print(f"Rows in sev_df: {num_rows_sev}")
print(f"Unique {ID_COL} in sev_df: {num_pols_unique_sev}")
print()
print(f"Total claims minus rows in sev_df (difference A): {total_claims - num_rows_sev}")
print(f"Policies with at least one claim minus unique {ID_COL} in sev_df (difference B): {num_pols_claimed - num_pols_unique_sev}")

Policies with no claim (ClaimNb = 0): 643953
Policies with at least one claim (ClaimNb > 0): 34060
Policies with multiple claims (ClaimNb > 1): 1882
Total claims (sum of ClaimNb): 36102

Rows in sev_df: 26639
Unique IDpol in sev_df: 24950

Total claims minus rows in sev_df (difference A): 9463
Policies with at least one claim minus unique IDpol in sev_df (difference B): 9110


**Insights**
- Both differences are positive (A: 9463, B: 9110). Therefore, the frequency table records both more claims and more claimed policies than the severity table has amounts for. Since these are only differences of totals, we check which policies are actually missing in the next section.

## 4. Matching Policies Across Tables

We now compare the policy IDs of the two tables directly to find out which policies appear in only one of them.

In [55]:
# Define a mask for checking if sev_ids are in freq_ids
in_freq = sev_ids.isin(freq_ids)

# Get the number of rows and distinct policies that are in sev_df but not freq_df
num_sev_rows_not_in_freq = int((~in_freq).sum())
num_sev_pols_not_in_freq = sev_ids[~in_freq].nunique()

# Get the number of policies with at least one claim that are not sev_df
num_pols_claimed_not_in_sev = int((~pols_claimed.isin(sev_ids)).sum())

# Get the number of policies with no claims that are in sev_df
num_pols_not_claimed_in_sev = int(pols_not_claimed.isin(sev_ids).sum())

# Print results
print(f"Rows in sev_df whose {ID_COL} is not in freq_df: {num_sev_rows_not_in_freq}")
print(f"Distinct {ID_COL} in sev_df that are not in freq_df: {num_sev_pols_not_in_freq}")
print(f"Policies with at least one claim that are not in sev_df: {num_pols_claimed_not_in_sev}")
print(f"Policies with no claim that are in sev_df: {num_pols_not_claimed_in_sev}")

Rows in sev_df whose IDpol is not in freq_df: 195
Distinct IDpol in sev_df that are not in freq_df: 6
Policies with at least one claim that are not in sev_df: 9116
Policies with no claim that are in sev_df: 0


As we can see, some claimed policies have no severity rows and a few severity rows belong to policies that are missing from the frequency table. Therefore, the two positive differences from section 3 are not only caused by claimed policies that are missing from the severity table but also from policies in the severity table that are missing from the frequency table. To identify the affected policies, we now count how many rows each policy of the frequency table has in the severity table and compare this number with `ClaimNb`. We then split the policies into matching, missing and partially matching ones and use these groups to rebuild difference A and difference B from the totals.

In [56]:
# Create a dataframe that shows only IDs and claim numbers
row_check = pd.DataFrame({
    ID_COL: freq_ids,
    "ClaimNb": freq_df["ClaimNb"],
})

# Add columns with the number of rows each ID has in sev_df and the gap to the number of claims
row_check["sev_rows"] = row_check[ID_COL].map(sev_id_counts).fillna(0).astype("int64")
row_check["gap"] = row_check["ClaimNb"] - row_check["sev_rows"]

# Define masks for IDs whose claims match, are missing or are partially missing in sev_df
matching = (row_check["ClaimNb"] > 0) & (row_check["sev_rows"] > 0) & (row_check["gap"] == 0)
missing = (row_check["ClaimNb"] > 0) & (row_check["sev_rows"] == 0)
partially_missing = (row_check["ClaimNb"] > 0) & (row_check["sev_rows"] > 0) & (row_check["gap"] != 0)

# Get the number of claims that are missing in sev_df and the net gap of partially missing claims
num_claims_missing = int(row_check.loc[missing, "ClaimNb"].sum())
gap_partially_missing = int(row_check.loc[partially_missing, "gap"].sum())

# Print results
print(f"Policies with at least one claim with matching {ID_COL} in sev_df: {matching.sum()}")
print(f"Policies with at least one claim without any {ID_COL} in sev_df: {missing.sum()}")
print(f"Policies with at least one claim with partially matching {ID_COL} in sev_df: {partially_missing.sum()}")
print()
print(f"Difference A (rebuilt):\n"
      f"  {total_claims} (total number of claims)\n"
      f"- {num_rows_sev} (number of sev_df rows)\n"
      f"{'-'*52}\n"
      f"= {num_claims_missing} (claims without sev_df rows)\n"
      f"+    {gap_partially_missing} (claims not fully covered by sev_df rows)\n"
      f"-  {num_sev_rows_not_in_freq} (rows in sev_df whose policy is not in freq_df)\n"
      f"{'-'*52}\n"
      f"= {num_claims_missing + gap_partially_missing - num_sev_rows_not_in_freq}")
print()
print(f"Difference B (rebuilt):\n"
      f"  {num_pols_claimed} (policies with at least one claim)\n"
      f"- {num_pols_unique_sev} (number of unique policies in sev_df)\n"
      f"{'-'*52}\n"
      f"= {num_pols_claimed_not_in_sev} (claimed policies not in sev_df)\n"
      f"-    {num_pols_not_claimed_in_sev} (unclaimed policies in sev_df)\n"
      f"-    {num_sev_pols_not_in_freq} (policies in sev_df not in freq_df)\n"
      f"{'-'*52}\n"
      f"= {num_pols_claimed_not_in_sev + num_pols_not_claimed_in_sev - num_sev_pols_not_in_freq}")

Policies with at least one claim with matching IDpol in sev_df: 24943
Policies with at least one claim without any IDpol in sev_df: 9116
Policies with at least one claim with partially matching IDpol in sev_df: 1

Difference A (rebuilt):
  36102 (total number of claims)
- 26639 (number of sev_df rows)
----------------------------------------------------
= 9657 (claims without sev_df rows)
+    1 (claims not fully covered by sev_df rows)
-  195 (rows in sev_df whose policy is not in freq_df)
----------------------------------------------------
= 9463

Difference B (rebuilt):
  34060 (policies with at least one claim)
- 24950 (number of unique policies in sev_df)
----------------------------------------------------
= 9116 (claimed policies not in sev_df)
-    0 (unclaimed policies in sev_df)
-    6 (policies in sev_df not in freq_df)
----------------------------------------------------
= 9110


Both rebuilt differences match the totals exactly and in both of them, the rows in sev_df whose policy is not in freq_df play a role. To understand which rows these are, the next cell lists the policies associated with these rows and checks whether the policies with the most rows in sev_df are among them.

In [57]:
# Get all IDs that are in sev_df but not in freq_df
sev_pols_not_in_freq = sev_ids[~in_freq]

# Get the 10 most frequent IDs in sev_df
top10_sev_pols= sev_id_counts.head(10).index

# Print results
freq_id_set = set(freq_ids)
print(f"Counts of policies in sev_df that are not in freq_df:\n{sev_pols_not_in_freq.value_counts()}")
print()
print(f"Top 10 most frequent policies in sev_df and whether they appear in freq_df:\n"
      f"{ {int(i): bool(i in freq_id_set) for i in top10_sev_pols} }")

Counts of policies in sev_df that are not in freq_df:
IDpol
2262511    66
2282134    36
2227533    25
2220367    24
2277846    23
2286775    21
Name: count, dtype: int64

Top 10 most frequent policies in sev_df and whether they appear in freq_df:
{2262511: False, 2282134: False, 2227533: False, 2220367: False, 2277846: False, 2286775: False, 2241683: True, 3253234: True, 3254353: True, 2248174: True}


**Insights**

- 9116 out of 34060 claimed policies that are in `freq_df` have no row in `sev_df`, which is six more than difference B computed in part 3. 
- Those six policies take up 195 rows in `sev_df` for which no corresponding row in `freq_df` exists. Thus, they have claim have amounts recorded in `sev_df` but no risk features and no exposure. Additionally, these six policies have far more rows than any policy in `freq_df` has claims (between 21 and 66 rows in `sev_df` compared to a maximum of 16 claims of any policy in `freq_df`).
- For all claimed policies that are in both `freq_df` and `sev_df` the number of rows in `sev_df` per policy equals `ClaimNb` (with the exception of one policy). Additionally, no policy without no claim has rows in `sev_df`. Therefore, a row in `sev_df` can be treated as corresponding to one claim, so that multiple rows per policy are normal and simply mean multiple claims associated with the same policy.
- On balance, while the `freq_df` can be used in full for modeling, `sev_df` can only be used for the 24944 policies found in both tables (after excluding the six policies that appear only in `sev_df` from this number).
- It is unknown why so many claims have no amount and whether this happens at random. This matters for the severity model later.

## 5. Distribution of Exposure and Claim Numbers

We now look at the value ranges of `Exposure` and at how often each claim number`ClaimNb` occurs to check their plausibility.

In [58]:
# Get the exposure per ID in freq_df
exposure = freq_df["Exposure"]

# Compute min, median, max of the exposure values
exposure_stats = exposure.agg(["min", "median", "max"])

# Count non-positive exposures and exposures above one
exposure_stats["count non-positive"] = (exposure <= 0).sum()
exposure_stats["count above one"] = (exposure > 1).sum()

# Get the absolute and relative counts of ClaimNb
claims_frequency = pd.DataFrame({
    "count": freq_df["ClaimNb"].value_counts(),
    "share": freq_df["ClaimNb"].value_counts(normalize=True),
}).sort_index()

# Get the IDs with CLaimNb above 4 including their exposures
pols_five_or_more_claims = (
    freq_df.loc[freq_df["ClaimNb"] > 4, [ID_COL, "ClaimNb", "Exposure"]]
    .sort_values("ClaimNb", ascending=False)
    )

# Print results
print(f"Exposure stats (values in years):\n{exposure_stats}")
print()
print(f"Frequency of claim numbers (ClaimNb):\n{claims_frequency}")
print()
print(f"Policies with claim numbers above 4 and their exposures:\n{pols_five_or_more_claims }")

Exposure stats (values in years):
min                      0.002732
median                   0.490000
max                      2.010000
count non-positive       0.000000
count above one       1224.000000
Name: Exposure, dtype: float64

Frequency of claim numbers (ClaimNb):
          count     share
ClaimNb                  
0        643953  0.949765
1         32178  0.047459
2          1784  0.002631
3            82  0.000121
4             7  0.000010
5             2  0.000003
6             1  0.000001
8             1  0.000001
9             1  0.000001
11            3  0.000004
16            1  0.000001

Policies with claim numbers above 4 and their exposures:
            IDpol  ClaimNb  Exposure
321248  2241683.0       16      0.33
6539      19471.0       11      1.00
488270  3254353.0       11      0.07
487268  3253234.0       11      0.08
321512  2248174.0        9      0.08
320764  2239279.0        8      0.41
304484  2216294.0        6      0.33
46470     93954.0        5      1.

**Insights**

- No `Exposure` value is zero or negative such that a division by `Exposure` or a log of `Exposure` is safe.
- The minimum `Exposure` of 0.002732 is about a day (1/365), which should be plausible.
- According to the OpenML dataset description, policies are "observed mostly on one year". However 1224 policies have an `Exposure` above 1 with a maximum 2.01. Since the description does not explain why some policies are observed for longer, this remains an open point.
- While the large majority of policies have at most one claim, nine policies have five or more claims and several of them combine many claims with a very short exposure (e.g., the 11 claims of `IDpol` 488270 are within 0.07 years, which means an average rate of a staggering 157 claims per year). Such values suggest data issues or unusual policies and may strongly influence our frequency model later.

## 6. Missing Values

Next, we count missing values and empty strings in each column of both `freq_df` and `sev_df` to check whether any data needs to be imputed or removed.

In [59]:
# Get missing values of all columns in both absolute and relative counts
missing_freq_abs = freq_df.isna().sum()
missing_freq_rel = freq_df.isna().mean()
missing_sev_abs = sev_df.isna().sum()
missing_sev_rel = sev_df.isna().mean()

# Get empty string values in both absolute and relative counts
empty_freq_abs = freq_df.isin([""]).sum()
empty_freq_rel = freq_df.isin([""]).mean()
empty_sev_abs = sev_df.isin([""]).sum()
empty_sev_rel = sev_df.isin([""]).mean()

# Merge both missing and empty values for freq_df in its own dataframe
missing_empty_freq = pd.DataFrame({
    "missing": missing_freq_abs,
    "share of missing":missing_freq_rel,
    "empty": empty_freq_abs,
    "share of empty": empty_freq_rel,
})

# Merge both missing and empty values for sev_df in its own dataframe
missing_empty_sev = pd.DataFrame({
    "missing": missing_sev_abs,
    "share of missing":missing_sev_rel,
    "empty": empty_sev_abs,
    "share of empty": empty_sev_rel,
})

# Print results
print(f"Missing and empty values in freq_df:\n{missing_empty_freq}")
print()
print(f"Missing and empty values in sev_df:\n{missing_empty_sev}")

Missing and empty values in freq_df:
            missing  share of missing  empty  share of empty
IDpol             0               0.0      0             0.0
ClaimNb           0               0.0      0             0.0
Exposure          0               0.0      0             0.0
Area              0               0.0      0             0.0
VehPower          0               0.0      0             0.0
VehAge            0               0.0      0             0.0
DrivAge           0               0.0      0             0.0
BonusMalus        0               0.0      0             0.0
VehBrand          0               0.0      0             0.0
VehGas            0               0.0      0             0.0
Density           0               0.0      0             0.0
Region            0               0.0      0             0.0

Missing and empty values in sev_df:
             missing  share of missing  empty  share of empty
IDpol              0               0.0      0             0.0
ClaimAmou

**Insights**

- neither `freq_df` nor `sev_df` has missing values or empty strings.
- Note that this check only covers real missing values and empty text, so that a value like 0 or 999 used as a placeholder in a numeric column would not show up here and has to be judged from the value ranges of the columns.
- No imputation of missing data is needed later.

## 7. Data Types and Column Contents

Finally, we check the data types and the actual values of the columns and note anything that could cause problems later.

In [60]:
# Get the summary stats for freq_df and sev_df
summary_stats_freq_no_id = freq_df.drop(columns=ID_COL).describe()
summary_stats_sev_no_id = sev_df.drop(columns=ID_COL).describe()

# Get the value counts for VehGas and ClaimAmount
veh_gas_value_counts = freq_df["VehGas"].value_counts()
claim_amount_value_counts = sev_df["ClaimAmount"].value_counts() 

# Get the number of categories of Area, VehBrand, and Region
cat_cols = ["Area", "VehBrand", "Region"]
area_brand_region = freq_df[cat_cols].nunique().to_frame("n_unique")

# Get the top 10 categories of Area, VehBrand, and Region
area_top5_cats = sorted(freq_df["Area"].astype(str).unique())[:5]
veh_brand_top5_cats = sorted(freq_df["VehBrand"].astype(str).unique())[:5]
region_top5_cats = sorted(freq_df["Region"].astype(str).unique())[:5]

# Print results
print(f"Summary statistics of freq_df:\n{summary_stats_freq_no_id.T[["mean", "std", "min", "50%", "max"]]}")
print()
print(f"Summary statistics of sev_df:\n{summary_stats_sev_no_id.T[["mean", "std", "min", "50%", "max"]]}")
print()
print(f"Counts of claim amounts:\n{veh_gas_value_counts}")
print()
print(f"Counts of vehicle gas types:\n{veh_gas_value_counts}")
print()
print(f"Unique categories in Area, VehBrand, and Region:\n{area_brand_region}")
print()
for cat_col, top5_cats in [("Area", area_top5_cats),("VehBrand", veh_brand_top5_cats), ("Region", region_top5_cats)]:
    print(f"Top 5 categories of {cat_col}: {top5_cats}")

Summary statistics of freq_df:
                   mean          std        min     50%       max
ClaimNb        0.053247     0.240117   0.000000    0.00     16.00
Exposure       0.528750     0.364442   0.002732    0.49      2.01
VehPower       6.454631     2.050906   4.000000    6.00     15.00
VehAge         7.044265     5.666232   0.000000    6.00    100.00
DrivAge       45.499122    14.137444  18.000000   44.00    100.00
BonusMalus    59.761502    15.636658  50.000000   50.00    230.00
Density     1792.422405  3958.646564   1.000000  393.00  27000.00

Summary statistics of sev_df:
                    mean           std  min     50%         max
ClaimAmount  2278.536382  29297.481336  1.0  1172.0  4075400.56

Counts of claim amounts:
VehGas
'Regular'    345877
'Diesel'     332136
Name: count, dtype: int64

Counts of vehicle gas types:
VehGas
'Regular'    345877
'Diesel'     332136
Name: count, dtype: int64

Unique categories in Area, VehBrand, and Region:
          n_unique
Area       

**Insights**

- `IDpol` has float64 in `freq_df` and int64 in `sev_df`, so a a merge or `isin` on mixed types can silently result in wrong matches. We need to case both to the same type later.
- `VehGas`'s values contain literal quote characters, i.e., `'Regular'` and `'Diesel'`. Thus, a filter like `== "Diesel"` would return no rows and we need to clean it later.
- `ClaimAmount` has a very heavy right tail as the largest claim is about 3500 times the median  and the standard deviation is far above the mean. Therefore, a few very large claims can dominate the total, which we need to take into acount later in the severity model.
- `VehAge` and `DrivAge` both reach a maximum of 100, while their medians are 6 and 44. These look like extreme or capped values.
- `Density` is strongly skewed as it has a median of 393, a mean of 1792, and a max of 27000. This may need a transformation for some models later.
- The median of `BonusMalus` equals its min of 50, so at least half of the policies sit at the lowest level.